In [ ]:
import json
from pathlib import Path

# 1. Resolve project root
PROJECT_ROOT = Path.cwd().parent

# 2. File paths
CHUNKED_DOCS = PROJECT_ROOT / "data" / "processed" / "chunked_documents.json"
INPUT_GT = PROJECT_ROOT / "data" / "evaluation" / "ground_truth.json"
OUTPUT_GT = PROJECT_ROOT / "data" / "evaluation" / "ground_truth_augmented.json"

def augment_dataset_from_json(
    chunks_path: Path = CHUNKED_DOCS,
    input_path: Path = INPUT_GT,
    output_path: Path = OUTPUT_GT,
):
    if not chunks_path.exists():
        raise FileNotFoundError(f"Chunked JSON not found at: {chunks_path}")

    # Load chunked documents
    with open(chunks_path, "r", encoding="utf-8") as f:
        chunks_data = json.load(f)

    # Build a fast lookup dictionary mapping chunk_id -> text content
    chunk_lookup = {}
    for item in chunks_data:
        # Handles various common keys for chunk IDs and text content
        c_id = item.get("chunk_id") or item.get("id")
        text = item.get("text") or item.get("content") or item.get("doc_json", "")
        
        if isinstance(text, dict):
            text = text.get("text", str(text))

        if c_id:
            chunk_lookup[c_id] = text

    print(f"📦 Loaded {len(chunk_lookup)} chunks into lookup dictionary.")

    # Load ground truth
    with open(input_path, "r", encoding="utf-8") as f:
        ground_truth = json.load(f)

    augmented_dataset = []
    missing_count = 0

    # Augment each record
    for item in ground_truth:
        target_id = item.get("target_chunk_id")
        chunk_text = chunk_lookup.get(target_id, "")

        if not chunk_text:
            missing_count += 1
            print(f"⚠️ Warning: chunk_id '{target_id}' not found in JSON.")

        augmented_item = {
            **item,
            "expected_output": chunk_text,
        }
        augmented_dataset.append(augmented_item)

    # Save output
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(augmented_dataset, f, indent=2, ensure_ascii=False)

    print(f"✅ Successfully augmented {len(augmented_dataset)} records.")
    print(f"📁 Saved to: {output_path}")

    if missing_count > 0:
        print(f"⚠️ Total missing chunks: {missing_count}")


# Run directly in notebook
augment_dataset_from_json()

In [ ]:
import asyncio
import os
from pathlib import Path

import pandas as pd
from openai import AsyncOpenAI
from tqdm.asyncio import tqdm

PROJECT_ROOT = Path.cwd().parent
AUGMENTED_GT = PROJECT_ROOT / "data" / "evaluation" / "ground_truth_augmented.json"
RESULTS_OUTPUT = PROJECT_ROOT / "data" / "evaluation" / "evaluation_results.json"

judge_client = AsyncOpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ.get("GROQ_API_KEY", "YOUR_GROQ_API_KEY"),
)

GROQ_JUDGE_MODEL = "llama-3.3-70b-versatile"
SEMAPHORE = asyncio.Semaphore(5)  # Controls concurrency to respect Groq rate limits

In [ ]:
JUDGE_PROMPT_FAITHFULNESS = """\
You are an expert evaluator assessing RAG system accuracy.
Evaluate if the Generated Answer is completely grounded in and supported ONLY by the Provided Context.

Context:
{context}

Generated Answer:
{answer}

Respond ONLY in valid JSON format with the following keys:
- "score": float between 0.0 and 1.0 (1.0 = completely faithful to context, 0.0 = contains hallucinations or external info)
- "reasoning": brief explanation of your score
"""

JUDGE_PROMPT_RELEVANCE = """\
You are an expert evaluator assessing response quality.
Evaluate if the Generated Answer directly answers the User Question.

User Question:
{question}

Generated Answer:
{answer}

Respond ONLY in valid JSON format with the following keys:
- "score": float between 0.0 and 1.0 (1.0 = directly answers the question concisely, 0.0 = off-topic or evasive)
- "reasoning": brief explanation of your score
"""

In [1]:
import os
from pathlib import Path
from groq import Groq
import sys

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.agent.agent import KnowledgeAgent
from src.retrieval.search_engines import KeywordSearchEngine
from src.tools.internal_search import InternalSearchTool
from src.tools.web_search import WebSearchTool  # Import your existing WebSearchTool
from src.tools.tool_registry import (
    ToolRegistry,
    INTERNAL_SEARCH_SCHEMA,
    WEB_SEARCH_SCHEMA,  # Import your existing WEB_SEARCH_SCHEMA
)

PROJECT_ROOT = Path.cwd().parent
DB_PATH = PROJECT_ROOT / "data" / "db" / "hnm.db"

# 1. Initialize Search Engines & Tools
search_engine = KeywordSearchEngine.from_db(db_path=str(DB_PATH))
internal_search_tool = InternalSearchTool(search_engine=search_engine)
web_search_tool = WebSearchTool(max_results=5) 

# 2. Setup Tool Registry
tool_registry = ToolRegistry()

# Define handlers that return formatted context strings
def handle_internal_search(query: str, num_results: int = 5) -> str:
    raw_results = internal_search_tool.search(query=query, num_results=num_results)
    return internal_search_tool.format_context(raw_results)

def handle_web_search(query: str, num_results: int = 5) -> str:
    raw_results = web_search_tool.search(query=query, num_results=num_results)
    return web_search_tool.format_context(raw_results)

# Register both tools with schema and matching handlers
tool_registry.register(INTERNAL_SEARCH_SCHEMA, handle_internal_search)
tool_registry.register(WEB_SEARCH_SCHEMA, handle_web_search)

# 3. Instantiate KnowledgeAgent
sync_groq_client = Groq(api_key=os.environ.get("GROQ_API_KEY"))
rag_agent = KnowledgeAgent(
    tool_registry=tool_registry,
    llm_client=sync_groq_client,
    model="qwen/qwen3.6-27b",
)

2026-08-09 20:32:21.853470866 [W:onnxruntime:Default, device_discovery.cc:134 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


In [2]:
rag_agent.ask("Who typically fills the Product Owner role?")

'In the Agile and Scrum framework, the **Product Owner (PO)** is best filled by an individual who has a deep understanding of both customer needs and business value, and the authority to make decisions about the product backlog.\n\nTypically, the role is filled by one of the following professionals:\n\n*   **Product Manager (PM):** This is the most common bridge. PMs already focus on the "why" and "what" of the product, aligning perfectly with the PO\'s responsibility to maximize value.\n*   **Business Analyst (BA):** In organizations where strategic vision is handled by leadership, a BA often steps in as the PO to translate high-level goals into detailed user stories and acceptance criteria.\n*   **Domain Expert / Subject Matter Expert (SME):** In specialized industries (e.g., fintech, healthcare), a domain expert may serve as the PO because they hold the crucial knowledge required to validate features accurately.\n*   **Internal Stakeholder or Customer Advocate:** Sometimes a senior 

In [3]:
rag_agent.messages

[{'role': 'system',
  'content': '\nYou are an intelligent internal Knowledge and Documentation Agent. Your job is to provide accurate, grounded, and helpful answers to user questions using available tools.\n\n---\n\n### YOUR AVAILABLE TOOLS:\n1. `search_internal_documentation`: Use this to search internal company documentation, technical guides, engineering standards, architecture specs, and private policies.\n2. `search_web`: Use this ONLY when the user\'s question explicitly asks about general public knowledge, external tech news, current public events, or third-party documentation not covered internally.\n\n---\n\n### TOOL USAGE GUIDELINES:\n- **Default Choice:** ALWAYS search internal documentation first (`search_internal_documentation`) when the query pertains to company processes, coding standards, tools, architecture, or project specs.\n- **Formulating Queries:** Pass concise, relevant keyword queries to the search tools rather than full conversational sentences.\n- **No Ground

In [4]:
import json
import time
from pathlib import Path

# Paths
PROJECT_ROOT = Path.cwd().parent
AUGMENTED_GT = (
    PROJECT_ROOT / "data" / "evaluation" / "ground_truth_augmented.json"
)
OUTPUT_FILE = (
    PROJECT_ROOT / "data" / "evaluation" / "agent_generated_answers.json"
)

# 1. Load ground truth dataset
with open(AUGMENTED_GT, "r", encoding="utf-8") as f:
    eval_dataset = json.load(f)

# Take 60 questions (or full dataset if <= 60)
questions_to_run = eval_dataset[:60]

results = []

# 2. Loop through and run agent
for i, item in enumerate(questions_to_run, start=1):
    question = item["question"]
    print(f"[{i}/{len(questions_to_run)}] Processing: {question}")

    try:
        answer = rag_agent.ask(question, clear_history=True)

        results.append(
            {
                "id": i,
                "question": question,
                "ground_truth": item.get("ground_truth", ""),
                "generated_answer": rag_agent.messages,
                "status": "success",
            }
        )
    except Exception as e:
        print(f"❌ Error on question #{i}: {e}")
        results.append(
            {
                "id": i,
                "question": question,
                "ground_truth": item.get("ground_truth", ""),
                "generated_answer": None,
                "status": "failed",
                "error": str(e),
            }
        )

    # Save progress incrementally after each question
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

    time.sleep(2)

print(f"\n✅ Finished! Results saved to: {OUTPUT_FILE}")

[1/60] Processing: Who typically fills the Product Owner role?


TypeError: Object of type ChatCompletionMessage is not JSON serializable